# NFL Play-by-Play Data Exploration

This notebook explores the play-by-play database schema and demonstrates the rich data available for analysis.

## Goal
Show MS students what's available in the database to inspire contributions and research projects.

## Data Source
- **nflverse** play-by-play data (1999-2025)
- 372+ fields including advanced analytics (EPA, WPA, CPOE)
- FTN charting data (2022+) - formations, personnel
- Snap counts (2012+) - player participation

In [ ]:
# Import required libraries
import pandas as pd
import sqlite3
from pathlib import Path
import sys

# Add src to path
src_path = Path.cwd().parent.parent / "src"
sys.path.insert(0, str(src_path))

from ffpy.database import FFPyDatabase
from ffpy.config import Config

# Connect to database
db = FFPyDatabase()
print(f"Database: {db.db_path}")
print(f"Connected successfully!")

## 1. Database Schema Overview

Let's explore what tables and columns are available.

In [ ]:
# Get all tables
tables_query = """
    SELECT name FROM sqlite_master 
    WHERE type='table' 
    ORDER BY name;
"""
tables = pd.read_sql(tables_query, db.conn)
print("Available tables:")
print(tables)

In [ ]:
# Get schema for plays table
plays_schema = pd.read_sql("PRAGMA table_info(plays)", db.conn)
print(f"\nPlays table has {len(plays_schema)} columns:")
print(plays_schema[["cid", "name", "type"]].to_string(index=False))

In [ ]:
# Get schema for games table
games_schema = pd.read_sql("PRAGMA table_info(games)", db.conn)
print(f"\nGames table has {len(games_schema)} columns:")
print(games_schema[["cid", "name", "type"]].to_string(index=False))

## 2. Data Statistics

How much data do we have?

In [ ]:
# Count records
stats_query = """
    SELECT 
        (SELECT COUNT(*) FROM plays) as total_plays,
        (SELECT COUNT(*) FROM games) as total_games,
        (SELECT COUNT(DISTINCT season) FROM plays) as seasons_loaded,
        (SELECT MIN(season) FROM plays) as earliest_season,
        (SELECT MAX(season) FROM plays) as latest_season
"""
stats = pd.read_sql(stats_query, db.conn)
print("\nDatabase Statistics:")
print(f"  Total Plays: {stats['total_plays'].iloc[0]:,}")
print(f"  Total Games: {stats['total_games'].iloc[0]:,}")
print(f"  Seasons: {stats['earliest_season'].iloc[0]} - {stats['latest_season'].iloc[0]}")
print(f"  Seasons Loaded: {stats['seasons_loaded'].iloc[0]}")

In [ ]:
# Plays by season
by_season = pd.read_sql(
    """
    SELECT 
        season,
        COUNT(*) as plays,
        COUNT(DISTINCT game_id) as games,
        COUNT(DISTINCT week) as weeks
    FROM plays
    GROUP BY season
    ORDER BY season DESC
""",
    db.conn,
)

print("\nPlays by Season:")
print(by_season.to_string(index=False))

## 3. Advanced Metrics Available

The database includes cutting-edge analytics metrics.

In [ ]:
# Sample advanced metrics from plays table
advanced_metrics = pd.read_sql(
    """
    SELECT 
        play_id,
        desc,
        epa,           -- Expected Points Added
        wpa,           -- Win Probability Added
        cpoe,          -- Completion % Over Expected
        air_epa,       -- EPA from air yards
        yac_epa,       -- EPA from yards after catch
        comp_air_epa,  -- Completed air EPA
        xyac_epa       -- Expected YAC EPA
    FROM plays
    WHERE play_type = 'pass'
        AND epa IS NOT NULL
    LIMIT 10
""",
    db.conn,
)

print("\nSample Advanced Metrics (passing plays):")
print(advanced_metrics)

## 4. Player Performance Examples

Let's look at some interesting player queries.

In [ ]:
# Top QBs by EPA in 2024 (minimum 100 dropbacks)
top_qbs = pd.read_sql(
    """
    SELECT 
        passer_player_name as player,
        COUNT(*) as dropbacks,
        ROUND(AVG(epa), 3) as avg_epa,
        ROUND(SUM(epa), 2) as total_epa,
        SUM(complete_pass) as completions,
        SUM(pass_touchdown) as tds,
        SUM(interception) as ints
    FROM plays
    WHERE season = 2024
        AND qb_dropback = 1
        AND passer_player_name IS NOT NULL
    GROUP BY passer_player_name
    HAVING dropbacks >= 100
    ORDER BY avg_epa DESC
    LIMIT 10
""",
    db.conn,
)

print("\nTop 10 QBs by EPA/play (2024):")
print(top_qbs.to_string(index=False))

In [ ]:
# Top receivers by target share (2024)
top_receivers = pd.read_sql(
    """
    WITH receiver_targets AS (
        SELECT 
            receiver_player_name,
            posteam,
            COUNT(*) as targets,
            SUM(complete_pass) as receptions,
            SUM(yards_gained) as yards,
            ROUND(AVG(air_yards), 1) as avg_air_yards,
            SUM(touchdown) as tds
        FROM plays
        WHERE season = 2024
            AND play_type = 'pass'
            AND receiver_player_name IS NOT NULL
        GROUP BY receiver_player_name, posteam
    ),
    team_targets AS (
        SELECT 
            posteam,
            COUNT(*) as team_targets
        FROM plays
        WHERE season = 2024
            AND play_type = 'pass'
        GROUP BY posteam
    )
    SELECT 
        r.receiver_player_name as player,
        r.posteam as team,
        r.targets,
        r.receptions,
        r.yards,
        r.tds,
        ROUND(100.0 * r.targets / t.team_targets, 1) as target_share_pct,
        r.avg_air_yards
    FROM receiver_targets r
    JOIN team_targets t ON r.posteam = t.posteam
    WHERE r.targets >= 40
    ORDER BY target_share_pct DESC
    LIMIT 15
""",
    db.conn,
)

print("\nTop 15 Receivers by Target Share (2024):")
print(top_receivers.to_string(index=False))

## 5. Situational Analysis

The database enables rich situational queries.

In [ ]:
# Red Zone efficiency by team (2024)
redzone = pd.read_sql(
    """
    SELECT 
        posteam as team,
        COUNT(*) as plays,
        SUM(CASE WHEN touchdown = 1 THEN 1 ELSE 0 END) as tds,
        ROUND(100.0 * SUM(touchdown) / COUNT(*), 1) as td_rate,
        ROUND(AVG(epa), 3) as avg_epa,
        SUM(CASE WHEN play_type = 'pass' THEN 1 ELSE 0 END) as passes,
        SUM(CASE WHEN play_type = 'run' THEN 1 ELSE 0 END) as runs
    FROM plays
    WHERE yardline_100 <= 20
        AND season = 2024
        AND play_type IN ('pass', 'run')
    GROUP BY posteam
    ORDER BY td_rate DESC
""",
    db.conn,
)

print("\nRed Zone Efficiency by Team (2024):")
print(redzone.to_string(index=False))

In [ ]:
# Third down conversion rates (2024)
third_downs = pd.read_sql(
    """
    SELECT 
        posteam as team,
        COUNT(*) as attempts,
        SUM(CASE WHEN first_down = 1 OR touchdown = 1 THEN 1 ELSE 0 END) as conversions,
        ROUND(100.0 * SUM(CASE WHEN first_down = 1 OR touchdown = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) as conversion_rate,
        ROUND(AVG(ydstogo), 1) as avg_distance,
        ROUND(AVG(epa), 3) as avg_epa
    FROM plays
    WHERE down = 3
        AND season = 2024
        AND play_type IN ('pass', 'run')
    GROUP BY posteam
    ORDER BY conversion_rate DESC
""",
    db.conn,
)

print("\nThird Down Conversion Rates (2024):")
print(third_downs.to_string(index=False))

## 6. Game Context Fields

Rich situational context is available for every play.

In [ ]:
# Sample play showing available context
context_example = pd.read_sql(
    """
    SELECT 
        game_id,
        game_date,
        week,
        posteam,
        defteam,
        down,
        ydstogo,
        yardline_100,
        quarter_seconds_remaining,
        half_seconds_remaining,
        game_seconds_remaining,
        score_differential,
        posteam_timeouts_remaining,
        defteam_timeouts_remaining,
        wp,              -- Win probability
        def_wp,          -- Defensive win probability
        home_wp,         -- Home team win probability
        away_wp,         -- Away team win probability
        desc
    FROM plays
    WHERE touchdown = 1
        AND season = 2024
    LIMIT 5
""",
    db.conn,
)

print("\nSample Plays with Context (Touchdowns):")
print(context_example.T)  # Transpose for readability

## 7. Player Identification Fields

Players are tracked with consistent IDs and positions.

In [ ]:
# Sample player ID fields
player_ids = pd.read_sql(
    """
    SELECT DISTINCT
        passer_player_id,
        passer_player_name,
        receiver_player_id,
        receiver_player_name,
        rusher_player_id,
        rusher_player_name
    FROM plays
    WHERE season = 2024
        AND (passer_player_name IS NOT NULL OR rusher_player_name IS NOT NULL)
    LIMIT 10
""",
    db.conn,
)

print("\nPlayer ID Fields Example:")
print(player_ids)

## 8. Research Ideas for MS Students

This database enables many interesting research projects:

### Machine Learning / Predictive Analytics
1. **Play Prediction**: Predict play type (run/pass) based on down, distance, score, time
2. **EPA Prediction**: Build models to predict EPA before the play
3. **Win Probability**: Improve win probability models using play-level features
4. **Player Performance**: Predict future player performance based on historical trends
5. **Injury Risk**: Analyze play characteristics and snap counts for injury prediction

### Statistical Analysis
6. **Situational Tendencies**: What plays work best in different situations?
7. **Coaching Decisions**: Analyze 4th down decisions, 2-point conversions
8. **Home Field Advantage**: Quantify home field impact by venue, team, weather
9. **Player Clustering**: Group players by playing style using advanced metrics
10. **Team Strategy Evolution**: How have team strategies changed over time?

### Optimization
11. **Play Calling Optimization**: Optimal play mix given situation
12. **Personnel Packages**: Which formations/personnel are most effective?
13. **Target Distribution**: Optimal target share for receivers
14. **Timeout Management**: When should teams use timeouts?

### Fantasy Football
15. **Projection Models**: Better player projections using play-level data
16. **Matchup Analysis**: Predict player performance vs specific defenses
17. **Usage Patterns**: Snap counts, target share, red zone opportunities
18. **DFS Optimization**: Daily fantasy lineup optimization with variance modeling

### Visualization
19. **Interactive Dashboards**: Team/player performance dashboards
20. **Play Visualization**: Animate plays or create play diagrams
21. **Trend Analysis**: Show how metrics evolve throughout season/career

## 9. Getting Started with Your Own Analysis

Here's a template for your own queries:

In [ ]:
# Template: Customize this for your analysis
your_query = """
    SELECT 
        -- Choose your fields
        season,
        week,
        posteam,
        play_type,
        yards_gained,
        epa
    FROM plays
    WHERE 1=1
        -- Add your filters
        AND season = 2024
        -- AND week = 1
        -- AND posteam = 'KC'
        -- AND play_type = 'pass'
    LIMIT 100
"""

# Run your query
results = pd.read_sql(your_query, db.conn)
print("\nYour Analysis Results:")
print(results.head(10))

## 10. Next Steps

1. **Explore the schema**: Run `PRAGMA table_info(plays)` to see all 372+ fields
2. **Try example queries**: Modify the queries above for different teams/players/seasons
3. **Join with other data**: Combine with rosters, betting lines, weather data
4. **Build models**: Export to CSV and use your favorite ML library
5. **Create visualizations**: Use plotly, matplotlib, or seaborn
6. **Contribute back**: Add new features to FFPy!

### Resources
- [nflverse Documentation](https://nflverse.nflverse.com/)
- [FFPy Database Guide](../../docs/db/DATABASE_GUIDE.md)
- [Play Analysis Example](../../examples/play_analysis_example.py)

In [ ]:
# Close database connection
db.close()
print("Database connection closed.")